# 04 — Pull NWSL Media Coverage Data

Pulls raw news-article metadata mentioning the NWSL from the MediaCloud API (US National collection), covering the league's full history to date.

> **Security note:** the original script had a live MediaCloud API key hardcoded in plaintext. Since this repo is public, that key is now read from an environment variable (or prompted for interactively) instead of being committed to the notebook. Set it once per session with:
> ```bash
> export MEDIACLOUD_API_KEY="your-key-here"
> ```
> or just run the notebook and enter it when prompted -- it won't be echoed or saved to the file.

This notebook does no filtering or cleaning -- that happens in `05_media_clean.ipynb`.

**Inputs:** none (pulls live from the MediaCloud API)
**Outputs:** `data/raw/nwsl_articles_raw.csv`


In [ ]:
import mediacloud.api
import pandas as pd
import os
import getpass
from datetime import datetime

DATA_RAW_DIR = os.path.join("..", "data", "raw")
os.makedirs(DATA_RAW_DIR, exist_ok=True)

MEDIACLOUD_API_KEY = os.environ.get("MEDIACLOUD_API_KEY") or getpass.getpass("MediaCloud API key: ")
mc_search = mediacloud.api.SearchApi(MEDIACLOUD_API_KEY)

US_NATIONAL_COLLECTION = 34412234

# NWSL launched in 2013, so start the search the day of its first match
start_date = datetime(2012, 11, 21)  # NWSL's first match (league announcement)
end_date = datetime(2024, 12, 31)    # update to your desired end date


In [ ]:
all_stories = []
pagination_token = None
more_stories = True

while more_stories:
    page, pagination_token = mc_search.story_list(
        "NWSL OR \"women's soccer\" OR \"National Women's Soccer League\"",
        collection_ids=[US_NATIONAL_COLLECTION],
        start_date=start_date,
        end_date=end_date,
        pagination_token=pagination_token,
    )
    all_stories += page
    more_stories = pagination_token is not None

print(f"Retrieved {len(all_stories)} matching stories")


In [ ]:
stories_data = [
    {
        "title": story.get("title"),
        "description": story.get("description"),
        "publish_date": story.get("publish_date"),
        "media_name": story.get("media_name"),
        "url": story.get("url"),
    }
    for story in all_stories
]

df = pd.DataFrame(stories_data)
df.to_csv(os.path.join(DATA_RAW_DIR, "nwsl_articles_raw.csv"), index=False)
print(f"Saved {len(df)} rows to nwsl_articles_raw.csv")
